# CHE1147 | Indoor PM₂.₅ prediction at 10-minute resolution

**Household-based comparison of linear regression and tree ensembles**

This notebook follows the original team's approach: aggregate the sensor readings to 10-minute intervals, construct historical predictors, and compare Linear Regression, Ridge, LightGBM and XGBoost. The baseline is the most recent indoor PM₂.₅ measurement.

**Prediction task.** At the start of a 10-minute interval, predict the **mean indoor PM₂.₅ concentration during that interval**, using observations from the *preceding completed* intervals. For example, measurements from 13:50–14:00 are used to predict the mean for 14:00–14:10. This is a **one-bin-ahead prediction**, not the one-hour-ahead task in the original proposal.

Changes from the team scripts are restricted to the parts needed for a reliable comparison: sources are joined by house and time, no current-target or future-label information is included in the inputs, preprocessing and tuning occur inside household-grouped CV, and the test households are not used for tuning. Model families and basic features remain familiar.

**Data:** LBNL / HENGH monitoring files in `Data/IAQ_Monitoring`, `Data/Airflow`, `Data/State_Monitoring` and optional `Data/Home_Equipment_Data`. **Outputs:** `Data/Results/PM25_10min_Colleague_Aligned_...`.

References within the dataset: `HENGH_IAQ_Monitoring_README.txt`, `HENGH_Airflow_README.txt` and `HENGH_State_Monitoring_README.txt`. Course topics: supervised regression, regularization, boosted trees, bias/variance and grouped validation.

## 1. Setup and analysis settings

Keep this notebook in `Data/Analysis`. Run the cells from top to bottom. The results folder is created automatically. `XGBoost` is optional because some Anaconda installations cannot load its OpenMP library; this should not prevent the remaining models from running.

The original team capped indoor PM₂.₅ at 100 µg/m³. The same setting is included below so the experiment is close to their approach. It is **not** a correction for sensor errors: capped outcomes must be labelled as such. Set `PM_CAP = None` for an uncapped sensitivity run; the results then go to a separate folder.

In [ ]:
from pathlib import Path
import json
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# If the notebook is open from Data/Analysis, its parent is Data.
LOCAL_DATA = Path('/Users/yazanzamel/Desktop/PhD/Courses/CHE1147/Data')
USER_DATA_ROOT = None  # Optional override: Path('/path/to/Data')
if USER_DATA_ROOT is not None:
    DATA_ROOT = Path(USER_DATA_ROOT).expanduser()
elif Path.cwd().name.lower() == 'analysis' and (Path.cwd().parent / 'IAQ_Monitoring').exists():
    DATA_ROOT = Path.cwd().parent
else:
    DATA_ROOT = LOCAL_DATA

RUN_MODE = 'full'      # 'audit' = load/check/EDA only; 'full' = also train/evaluate
FREQ = '10min'
PM_CAP = 100.0         # Original team setting; use None to keep all nonnegative readings
MIN_PM_POINTS = 5      # At least 5 minute readings in each accepted 10-min PM bin
FEATURE_LEVEL = 'medium'   # 'low', 'medium' or 'high'
TEST_FRACTION = 0.20
RANDOM_STATE = 2026     # Different deterministic household split from original script (42)
CV_FOLDS = 5
N_RANDOM_SEARCH = 10    # Number of parameter combinations for each tuned model
MIN_HOMES_FOR_MODEL = 12
USE_LIGHTGBM = True
USE_XGBOOST = True       # Automatically skipped if the library cannot import

suffix = 'cap100' if PM_CAP == 100.0 else ('raw' if PM_CAP is None else f'cap{PM_CAP:g}')
RESULTS = DATA_ROOT / 'Results' / f'PM25_10min_Colleague_Aligned_{suffix}'
TABLES, FIGURES, MODELS = (RESULTS / x for x in ('tables', 'figures', 'models'))
for folder in (TABLES, FIGURES, MODELS):
    folder.mkdir(parents=True, exist_ok=True)

assert (DATA_ROOT / 'IAQ_Monitoring').is_dir(), f'IAQ_Monitoring not found under {DATA_ROOT}'
assert RUN_MODE in ('audit', 'full')
assert FEATURE_LEVEL in ('low', 'medium', 'high')
print('Data folder:   ', DATA_ROOT)
print('Results folder:', RESULTS)
print('Prediction:    next 10-minute bin (based on completed earlier bins)')
print('PM treatment:  ', f'cap at {PM_CAP:g} µg/m³' if PM_CAP is not None else 'uncapped nonnegative PM')

## 2. Read the monitoring files by household

The household ID comes from the filename. The air-quality measurements are the base time series; airflow and state-monitoring measurements are joined by the same 10-minute bin. The data are **not** stacked across sensor types.

The HENGH README recommends averaging noisy outdoor PM measurements over at least 10 minutes. We record the number of one-minute indoor PM samples and exclude bins with fewer than five readings. Missing sensor measurements stay missing until the model's training-fold imputer handles them. We do not interpolate indoor PM targets.

In [ ]:
SOURCES = {
    'IAQ_Monitoring': 'DataTable',
    'Airflow': 'Qtotal',
    'State_Monitoring': 'StateTable',
}

def find_household_files(subfolder, prefix):
    folder = DATA_ROOT / subfolder
    result = {}
    if not folder.is_dir():
        return result
    for path in sorted(folder.glob('*.csv')):
        match = re.fullmatch(rf'{re.escape(prefix)}(\d{{3}})', path.stem, flags=re.I)
        if match:
            result[match.group(1)] = path
    return result

files = {s: find_household_files(s, p) for s, p in SOURCES.items()}
for folder, inventory in files.items():
    print(f'{folder:19} {len(inventory):3} household files')

if not files['IAQ_Monitoring']:
    raise FileNotFoundError('Expected DataTable###.csv files in IAQ_Monitoring')

HOME_FIELDS = {
    'YearBuilt': 'year_built',
    'N_Stories': 'n_stories',
    'FloorArea_sqft': 'floor_area_sqft',
    'CeilingHgt_ft': 'ceiling_height_ft',
}
metadata_path = DATA_ROOT / 'Home_Equipment_Data' / 'HENGH_Study_Home_Characteristics.xlsx'
if metadata_path.is_file():
    home_table = pd.read_excel(metadata_path, sheet_name='HENGH_House_Data')
    home_table['household_id'] = pd.to_numeric(home_table['Home_ID'], errors='coerce').astype('Int64').astype(str).str.zfill(3)
    present = [c for c in HOME_FIELDS if c in home_table.columns]
    home_metadata = (home_table[['household_id'] + present].rename(columns=HOME_FIELDS)
                     .drop_duplicates(subset=['household_id']).set_index('household_id'))
    print('Home characteristics: ', len(home_metadata), 'rows')
else:
    home_metadata = pd.DataFrame(columns=list(HOME_FIELDS.values()))
    print('No home characteristics workbook. Continuing without static fields.')

In [ ]:
def read_binned_source(path):
    """Read minute-level source data and aggregate to 10-minute bins, labelled by bin start."""
    raw = pd.read_csv(path, low_memory=False, encoding='utf-8-sig')
    raw.columns = raw.columns.astype(str).str.strip()
    if 'Time' not in raw.columns:
        raise ValueError(f'{path.name}: missing Time column')

    raw['timestamp'] = pd.to_datetime(raw.pop('Time'), errors='coerce')
    bad_timestamps = int(raw.timestamp.isna().sum())
    raw = raw.dropna(subset=['timestamp']).sort_values('timestamp').set_index('timestamp')
    for col in raw.columns:
        raw[col] = pd.to_numeric(raw[col], errors='coerce')

    binned = raw.resample(FREQ, closed='left', label='left').mean(numeric_only=True)
    counts = raw.notna().resample(FREQ, closed='left', label='left').sum()
    binned.index.name = 'timestamp'
    return binned, counts, bad_timestamps


def summarize_state(state):
    """Combine variable sensor layouts into three comparable activity proxies."""
    doors = [c for c in state if c.startswith('STN_')]
    burners = [c for c in state if c.startswith('IBU_C')]
    ovens = [c for c in state if c.startswith('IBU_OVN')]
    s = pd.DataFrame(index=state.index)
    s['door_open_pct_max'] = state[doors].max(axis=1) if doors else np.nan
    s['cooktop_temp_max'] = state[burners].max(axis=1) if burners else np.nan
    s['oven_temp_max'] = state[ovens].max(axis=1) if ovens else np.nan
    return s


def combine_one_home(hid, iaq_path):
    iaq, counts, bad_timestamps = read_binned_source(iaq_path)
    if 'PM_MET_IN1' not in iaq.columns:
        iaq['PM_MET_IN1'] = np.nan

    # Inspect and retain a count of suspicious readings rather than silently deleting them.
    original_pm = iaq['PM_MET_IN1'].copy()
    negative_bins = int((original_pm < 0).sum())
    high_bins = int((original_pm > 100).sum())
    iaq.loc[iaq.PM_MET_IN1 < 0, 'PM_MET_IN1'] = np.nan
    counts_pm = counts['PM_MET_IN1'] if 'PM_MET_IN1' in counts else pd.Series(0, index=iaq.index)
    iaq.loc[counts_pm < MIN_PM_POINTS, 'PM_MET_IN1'] = np.nan
    high_eligible = int((iaq.PM_MET_IN1 > 100).sum())
    if PM_CAP is not None:
        iaq['PM_MET_IN1'] = iaq['PM_MET_IN1'].clip(upper=PM_CAP)

    # Apply the count requirement to outdoor PM as well: the README notes minute-scale noise.
    if 'PM_MET_OUT' in iaq.columns:
        iaq.loc[iaq.PM_MET_OUT < 0, 'PM_MET_OUT'] = np.nan
        out_count = counts['PM_MET_OUT'] if 'PM_MET_OUT' in counts else pd.Series(0, index=iaq.index)
        iaq.loc[out_count < MIN_PM_POINTS, 'PM_MET_OUT'] = np.nan

    combined = iaq.copy()
    flow_path = files['Airflow'].get(hid)
    state_path = files['State_Monitoring'].get(hid)
    if flow_path is not None:
        flow, _, _ = read_binned_source(flow_path)
        flow = flow.drop(columns=combined.columns.intersection(flow.columns), errors='ignore')
        combined = combined.join(flow, how='left')
    if state_path is not None:
        state, _, _ = read_binned_source(state_path)
        combined = combined.join(summarize_state(state), how='left')

    if hid in home_metadata.index:
        for c in HOME_FIELDS.values():
            if c in home_metadata.columns:
                combined[c] = pd.to_numeric(home_metadata.loc[hid, c], errors='coerce')

    combined['household_id'] = hid
    # The bin indexed t is the prediction TARGET. No same-bin measurements are features.
    # Lag 1 = last COMPLETED 10-minute bin, available when bin t begins.
    history = {
        'PM_MET_IN1': (1, 3, 6, 12, 18, 36, 72, 144),
        'PM_MET_OUT': (1, 6),
        'ACH': (1, 3),
        'QCFM_Total': (1, 3),
    }
    other_last_bin = [
        'QCFM_Exhaust', 'QCFM_Supply', 'QCFM_Infiltration', 'RHD_CFM',
        'CO2_ETC_IN1', 'T_ETC_IN1', 'RH_ETC_IN1',
        'T_ETC_OUT', 'RH_ETC_OUT', 'T_HBO_OUT', 'RH_HBO_OUT',
        'door_open_pct_max', 'cooktop_temp_max', 'oven_temp_max',
    ]
    for sensor, offsets in history.items():
        if sensor in combined:
            for offset in offsets:
                combined[f'{sensor}_lag{offset}'] = combined[sensor].shift(offset)
    for sensor in other_last_bin:
        if sensor in combined:
            combined[f'{sensor}_lag1'] = combined[sensor].shift(1)

    combined['hour'] = combined.index.hour
    combined['dayofweek'] = combined.index.dayofweek
    combined['is_weekend'] = (combined.index.dayofweek >= 5).astype(int)
    combined['month'] = combined.index.month
    combined['target_pm'] = combined['PM_MET_IN1']

    # Require both a measured target and the last measured PM value for fair persistence comparison.
    valid = combined['target_pm'].notna() & combined['PM_MET_IN1_lag1'].notna()
    audit_row = {
        'household_id': hid,
        'available_10min_bins': int(len(combined)),
        'measured_pm_bins': int(combined.PM_MET_IN1.notna().sum()),
        'eligible_rows': int(valid.sum()),
        'invalid_iaq_timestamps': bad_timestamps,
        'pm_bins_negative_raw': negative_bins,
        'pm_bins_above_100_raw': high_bins,
        'pm_bins_above_100_after_sample_requirement': high_eligible,
        'airflow_file_found': flow_path is not None,
        'state_file_found': state_path is not None,
    }
    return combined.reset_index(), audit_row

## 3. Assemble the eligible modeling records

Each record corresponds to one target 10-minute interval. An observation is eligible when both the target and the immediately preceding indoor PM₂.₅ measurement exist. The latter is necessary for a fair comparison against persistence. We keep the full source inventory and missingness audit so exclusions are explicit.

In [ ]:
all_homes, audit_rows = [], []
for hid, path in files['IAQ_Monitoring'].items():
    try:
        h, details = combine_one_home(hid, path)
        all_homes.append(h)
        audit_rows.append(details)
    except Exception as exc:
        print(f'Home {hid}: {exc}')
        audit_rows.append({'household_id':hid, 'eligible_rows':0, 'load_error':str(exc)})

if not all_homes:
    raise RuntimeError('No household data could be read')

audit = pd.DataFrame(audit_rows).sort_values('household_id')
audit.to_csv(TABLES / '01_household_audit.csv', index=False)
all_data = pd.concat(all_homes, ignore_index=True, sort=False)
all_data = all_data.sort_values(['household_id','timestamp']).reset_index(drop=True)
eligible = all_data.loc[all_data['target_pm'].notna() &
                        all_data['PM_MET_IN1_lag1'].notna()].copy()
print('Homes with usable targets:', eligible.household_id.nunique())
print('Eligible 10-min records:  ', f'{len(eligible):,}')
print('Raw 10-min PM bins >100: ', int(audit['pm_bins_above_100_raw'].fillna(0).sum()))
display(audit.head(10))

if eligible.household_id.nunique() < 3:
    raise ValueError('At least three target-eligible households are needed for a grouped split')

## 4. Select the inputs

The original team used low/medium/high feature priorities. This keeps that arrangement, except each time-varying sensor measurement is moved to its **previous completed interval** (`_lag1`) so the model cannot use information from the target interval itself. `hh_mean_pm` is omitted because its training-house calculation uses future outcomes and cannot be reproduced for a new home. The external daily ambient-PM workbook is not joined without a verified time-availability rule.

All columns are selected using the **development data only**, after the household split. Missing predictors are handled by median imputation inside cross-validation, rather than before the split.

In [ ]:
PRIORITY_1 = [
    'PM_MET_IN1_lag1','PM_MET_IN1_lag3','PM_MET_IN1_lag6','PM_MET_IN1_lag12',
    'PM_MET_OUT_lag1','ACH_lag1','QCFM_Total_lag1',
    'CO2_ETC_IN1_lag1','T_ETC_IN1_lag1','RH_ETC_IN1_lag1',
    'hour','is_weekend',
]
PRIORITY_2 = [
    'PM_MET_IN1_lag18','PM_MET_IN1_lag36','PM_MET_IN1_lag72',
    'QCFM_Exhaust_lag1','QCFM_Supply_lag1','QCFM_Infiltration_lag1','RHD_CFM_lag1',
    'T_ETC_OUT_lag1','RH_ETC_OUT_lag1','T_HBO_OUT_lag1','RH_HBO_OUT_lag1',
    'PM_MET_OUT_lag6','QCFM_Total_lag3','ACH_lag3',
    'door_open_pct_max_lag1','cooktop_temp_max_lag1','oven_temp_max_lag1',
    'dayofweek','month','floor_area_sqft','n_stories','year_built','ceiling_height_ft',
]
PRIORITY_3 = ['PM_MET_IN1_lag144']
feature_candidates = PRIORITY_1.copy()
if FEATURE_LEVEL in ('medium','high'):
    feature_candidates += PRIORITY_2
if FEATURE_LEVEL == 'high':
    feature_candidates += PRIORITY_3

from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import RandomizedSearchCV, cross_val_score

splitter = GroupShuffleSplit(n_splits=1, test_size=TEST_FRACTION, random_state=RANDOM_STATE)
train_idx, test_idx = next(splitter.split(eligible, groups=eligible.household_id))
dev = eligible.iloc[train_idx].copy()
test = eligible.iloc[test_idx].copy()
assert set(dev.household_id).isdisjoint(set(test.household_id))

features = [c for c in feature_candidates if c in dev and dev[c].notna().any()]
assert 'PM_MET_IN1_lag1' in features
assert 'PM_MET_IN1' not in features and 'target_pm' not in features
X_dev, y_dev = dev[features], dev['target_pm']
X_test, y_test = test[features], test['target_pm']
groups = dev.household_id

missingness = (dev[features].isna().mean() * 100).sort_values(ascending=False)
missingness.rename('development_missing_pct').to_csv(TABLES / '02_feature_missingness.csv')

split_summary = {
    'eligible_homes': int(eligible.household_id.nunique()),
    'development_homes': int(dev.household_id.nunique()),
    'held_out_homes': int(test.household_id.nunique()),
    'development_rows': int(len(dev)),
    'held_out_rows': int(len(test)),
    'features_used': int(len(features)),
}
pd.DataFrame([split_summary]).to_csv(TABLES / '03_split_summary.csv', index=False)
pd.DataFrame({'household_id': sorted(dev.household_id.unique())}).to_csv(TABLES/'03_development_home_ids.csv',index=False)
pd.DataFrame({'household_id': sorted(test.household_id.unique())}).to_csv(TABLES/'03_test_home_ids.csv',index=False)
print('Split:', split_summary)
print('Selected predictors:')
print(', '.join(features))

## 5. EDA on development homes only

A quick check before fitting: concentration distribution, correlations with the target, and autocorrelation in five individual homes. The test homes remain out of these diagnostics. The histogram hides the top 1% **for display only**; the actual observations are not dropped. The feature correlation plot describes associations, not causal effects.

In [ ]:
# Development-only distribution of the 10-minute target.
fig, ax = plt.subplots(figsize=(7.2,4.2))
vals = dev.target_pm.dropna()
cutoff = float(vals.quantile(.99))
ax.hist(vals[vals <= cutoff], bins=36, edgecolor='white', linewidth=.3)
ax.set(xlabel='Indoor PM₂.₅ in target bin (µg/m³)', ylabel='Number of intervals',
       title='Development homes: indoor PM₂.₅ distribution')
ax.text(.98,.96, f'Plot-only upper 1% omitted: {(vals > cutoff).sum():,} bins',
        transform=ax.transAxes, ha='right', va='top', fontsize=9)
fig.tight_layout();fig.savefig(FIGURES/'01_target_distribution.png', dpi=200);plt.show()

# Correlations: predictors and next-bin target. Show the strongest 9 predictors.
correlations = dev[features].corrwith(dev.target_pm).dropna().sort_values(key=lambda x:x.abs(), ascending=False)
ranked = correlations.head(9).index.tolist()
subset = ranked + ['target_pm']
corr_matrix = dev[subset].corr()
fig, ax = plt.subplots(figsize=(8,6.4))
heat = ax.imshow(corr_matrix, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(subset)), subset, rotation=70, ha='right')
ax.set_yticks(range(len(subset)), subset)
ax.set_title('Development homes: correlations with next-bin PM₂.₅')
fig.colorbar(heat, ax=ax, label='Pearson correlation')
fig.tight_layout();fig.savefig(FIGURES/'02_development_correlations.png',dpi=200,bbox_inches='tight');plt.show()
correlations.rename('pearson_r').to_csv(TABLES/'02_feature_target_correlations.csv')

# Autocorrelation on regular bins for individual development homes.
fig,ax=plt.subplots(figsize=(8,4.2))
long_homes = (dev.groupby('household_id').size().sort_values(ascending=False).head(5).index)
for hid in long_homes:
    series = all_data.loc[all_data.household_id == hid, ['timestamp','PM_MET_IN1']].drop_duplicates('timestamp')
    series = series.set_index('timestamp')['PM_MET_IN1'].sort_index()
    lags = list(range(1,19))
    ac = [series.autocorr(lag=k) for k in lags]
    ax.plot(np.array(lags)*10, ac, marker='.', linewidth=1.5, label=f'Home {hid}')
ax.set(xlabel='Time lag (minutes)',ylabel='Autocorrelation',title='Indoor PM₂.₅ temporal dependence')
ax.legend(fontsize=8)
fig.tight_layout();fig.savefig(FIGURES/'03_development_autocorrelation.png',dpi=200);plt.show()

## 6. Model development and grouped validation

Models remain the same as in the original team scripts. We compare a persistence benchmark, ordinary least squares, Ridge, LightGBM, and—if it imports—XGBoost. The primary selection metric is **mean absolute error (MAE)**. We use five-fold `GroupKFold` on development homes: measurements from one household cannot appear in both training and validation within a fold.

`SimpleImputer` and (for linear models) `StandardScaler` sit inside a scikit-learn `Pipeline`, so their fitted statistics are learned separately within each training fold. The final test set is not consulted by `RandomizedSearchCV` or used for early stopping. Hyperparameter tuning is deliberately modest; this is a course-level model comparison, not an unrestricted search.

In [ ]:
def metrics(y, pred):
    return {
        'MAE': float(mean_absolute_error(y,pred)),
        'RMSE': float(np.sqrt(mean_squared_error(y,pred))),
        'R2': float(r2_score(y,pred)),
    }

def build_pipeline(estimator, scale=False):
    steps = [('imputer', SimpleImputer(strategy='median', keep_empty_features=True))]
    if scale:
        steps.append(('scaler',StandardScaler()))
    steps.append(('model',estimator))
    return Pipeline(steps)

linear = build_pipeline(LinearRegression(), scale=True)
ridge = build_pipeline(Ridge(), scale=True)
spaces = {
    'Ridge': (ridge, {'model__alpha':[.1,1.,10.,100.,1000.]}),
}

if USE_LIGHTGBM:
    try:
        from lightgbm import LGBMRegressor
        lightgbm_model = build_pipeline(LGBMRegressor(random_state=RANDOM_STATE,n_jobs=1,verbose=-1))
        spaces['LightGBM'] = (lightgbm_model, {
            'model__n_estimators':[150,300,500,750],
            'model__learning_rate':[.02,.04,.07],
            'model__num_leaves':[7,15,31],
            'model__max_depth':[3,5,7],
            'model__min_child_samples':[20,40,70],
        })
        print('LightGBM available')
    except Exception as exc:
        print('LightGBM unavailable; continuing without it:', str(exc).splitlines()[0])

if USE_XGBOOST:
    try:
        from xgboost import XGBRegressor
        xgboost_model = build_pipeline(XGBRegressor(
            random_state=RANDOM_STATE,n_jobs=1,tree_method='hist',
            objective='reg:squarederror',verbosity=0))
        spaces['XGBoost'] = (xgboost_model, {
            'model__n_estimators':[150,300,500,750],
            'model__learning_rate':[.02,.04,.07],
            'model__max_depth':[2,3,5],
            'model__min_child_weight':[1,6,15],
            'model__subsample':[.7,.85,1.],
        })
        print('XGBoost available')
    except Exception as exc:
        print('XGBoost unavailable; continuing without it:', str(exc).splitlines()[0])

print('Ready:', ['Persistence','Linear'] + list(spaces))

In [ ]:
RUN_TRAINING = (RUN_MODE == 'full' and eligible.household_id.nunique() >= MIN_HOMES_FOR_MODEL)
fitted, fold_records, tuned = {}, [], {}
BEST_MODEL = None

if not RUN_TRAINING:
    print('Training skipped (audit mode or fewer than', MIN_HOMES_FOR_MODEL, 'eligible homes).')
else:
    folds = min(CV_FOLDS, dev.household_id.nunique())
    cv = GroupKFold(n_splits=folds)
    # Persistence is scored on each validation fold without fitting.
    for i, (_, v) in enumerate(cv.split(X_dev,y_dev,groups), 1):
        m = metrics(y_dev.iloc[v], dev['PM_MET_IN1_lag1'].iloc[v])
        fold_records.append({'model':'Persistence','fold':i,**m})

    # Untuned linear regression.
    linear_mae = cross_val_score(linear,X_dev,y_dev,groups=groups,cv=cv,
                                 scoring='neg_mean_absolute_error',n_jobs=1)
    fold_records.extend({'model':'Linear','fold':k,'MAE':float(-v)}
                        for k,v in enumerate(linear_mae,1))
    linear.fit(X_dev,y_dev)
    fitted['Linear'] = linear

    for name,(estimator,param_space) in spaces.items():
        grid_size = int(np.prod([len(v) for v in param_space.values()]))
        search = RandomizedSearchCV(
            estimator, param_space,
            n_iter=min(N_RANDOM_SEARCH, grid_size), cv=cv,
            scoring='neg_mean_absolute_error',n_jobs=1,
            random_state=RANDOM_STATE,refit=True,error_score='raise')
        search.fit(X_dev,y_dev,groups=groups)
        fitted[name] = search.best_estimator_
        tuned[name] = search.best_params_
        for k in range(folds):
            fold_records.append({'model':name, 'fold':k+1,
                                 'MAE':float(-search.cv_results_[f'split{k}_test_score'][search.best_index_])})
        print(f'{name:10s} CV MAE: {-search.best_score_:.3f} µg/m³')

    cv_details = pd.DataFrame(fold_records)
    cv_details.to_csv(TABLES/'04_grouped_cv_folds.csv',index=False)
    cv_summary = (cv_details.groupby('model',as_index=False)['MAE']
                  .agg(CV_MAE='mean',CV_MAE_SD='std').sort_values('CV_MAE'))
    cv_summary.to_csv(TABLES/'05_grouped_cv_summary.csv',index=False)
    (TABLES/'06_selected_parameters.json').write_text(json.dumps(tuned,indent=2),encoding='utf-8')
    BEST_MODEL = str(cv_summary.iloc[0]['model'])
    print('Selected from development CV:',BEST_MODEL)
    display(cv_summary.round(3))

## 7. Final evaluation: previously unseen households

Only now do we predict the held-out homes. All available models use the exact same target observations. The metrics include pooled MAE, RMSE and R² as well as the mean per-home MAE, because homes contribute different numbers of observations.

A model selected on development CV can perform worse than another model on the held-out set; **do not change the selected model after looking at test MAE**. Since earlier iterations of this project already examined held-out homes, these results should be described as a retrospective household-holdout comparison rather than a completely untouched external validation.

In [ ]:
if RUN_TRAINING:
    predictions = {'Persistence':test['PM_MET_IN1_lag1'].to_numpy()}
    for name, estimator in fitted.items():
        predictions[name] = estimator.predict(X_test)

    by_prediction = test[['household_id','timestamp','target_pm']].copy()
    by_prediction = by_prediction.rename(columns={'timestamp':'target_bin_start','target_pm':'actual_pm'})
    for name, pred in predictions.items():
        by_prediction[f'pred_{name.lower()}'] = np.asarray(pred)
    by_prediction.to_csv(TABLES/'08_held_out_predictions.csv',index=False)

    test_metrics, home_rows = [], []
    for name, pred in predictions.items():
        overall = metrics(y_test,pred)
        frame = pd.DataFrame({
            'household_id':test.household_id.to_numpy(),
            'absolute_error':np.abs(y_test.to_numpy()-np.asarray(pred))})
        home_mae = frame.groupby('household_id')['absolute_error'].mean()
        test_metrics.append({'Model':name,**overall,
                             'Mean_home_MAE':float(home_mae.mean())})
        for hid, val in home_mae.items():
            home_rows.append({'household_id':hid,'Model':name,'MAE':float(val)})

    test_results = pd.DataFrame(test_metrics).sort_values('MAE').reset_index(drop=True)
    home_errors = pd.DataFrame(home_rows)
    test_results.to_csv(TABLES/'07_held_out_test_metrics.csv',index=False)
    home_errors.to_csv(TABLES/'09_per_home_errors.csv',index=False)
    display(test_results.round(3))

    fig,ax=plt.subplots(figsize=(7.5,4.3))
    sorted_result = test_results.sort_values('MAE',ascending=False)
    ax.barh(sorted_result.Model,sorted_result.MAE)
    ax.set(xlabel='Held-out MAE (µg/m³)',title='One-bin prediction: comparison on held-out homes')
    fig.tight_layout();fig.savefig(FIGURES/'04_test_model_comparison.png',dpi=200);plt.show()

    # A separate plot for the best *trained* model, even if persistence wins CV.
    trained_rank = cv_summary.loc[cv_summary.model != 'Persistence','model'].tolist()
    best_trained = trained_rank[0] if trained_rank else 'Persistence'
    fig,ax=plt.subplots(figsize=(6,5))
    ax.scatter(y_test, predictions[best_trained],s=8,alpha=.23)
    lim = max(float(y_test.max()),float(np.max(predictions[best_trained])),1.)
    ax.plot([0,lim],[0,lim],'k--',lw=1)
    ax.set(xlabel='Observed indoor PM₂.₅ (µg/m³)',ylabel='Predicted indoor PM₂.₅ (µg/m³)',
           title=f'{best_trained}: predictions for held-out homes')
    fig.tight_layout();fig.savefig(FIGURES/'05_best_trained_pred_vs_actual.png',dpi=200);plt.show()

    fig,ax=plt.subplots(figsize=(8.5,4.4))
    selected_home_err = home_errors[home_errors.Model == BEST_MODEL].sort_values('MAE')
    ax.bar(selected_home_err.household_id.astype(str),selected_home_err.MAE)
    ax.set(xlabel='Held-out household ID',ylabel='MAE (µg/m³)',title=f'Per-household error: {BEST_MODEL}')
    ax.tick_params(axis='x',rotation=45)
    fig.tight_layout();fig.savefig(FIGURES/'06_error_by_household.png',dpi=200);plt.show()

    example_home = by_prediction.household_id.value_counts().idxmax()
    view = by_prediction[by_prediction.household_id==example_home].sort_values('target_bin_start')
    fig,ax=plt.subplots(figsize=(10.5,4.0))
    ax.plot(view.target_bin_start,view.actual_pm,label='Actual',lw=1.1)
    ax.plot(view.target_bin_start,view[f'pred_{best_trained.lower()}'],
            label=f'{best_trained}',lw=1.0,alpha=.85)
    ax.plot(view.target_bin_start,view.pred_persistence,
            label='Persistence',lw=.9,alpha=.75)
    ax.set(xlabel='Target-bin time',ylabel='Indoor PM₂.₅ (µg/m³)',
           title=f'Unseen home {example_home}: next-interval prediction')
    ax.legend();fig.autofmt_xdate();fig.tight_layout()
    fig.savefig(FIGURES/'07_unseen_home_time_series.png',dpi=200);plt.show()

    # Importance is the tree model's built-in split/weight measure, not causation.
    for name in ('LightGBM','XGBoost'):
        estimator = fitted.get(name)
        if estimator is None:
            continue
        imp = pd.Series(estimator.named_steps['model'].feature_importances_,index=features)
        imp = imp.sort_values(ascending=False)
        imp.rename('importance').to_csv(TABLES/f'10_{name.lower()}_feature_importance.csv')
        fig,ax=plt.subplots(figsize=(8,4.5))
        imp.head(12).sort_values().plot(kind='barh',ax=ax)
        ax.set(xlabel='Built-in model importance (not causal)',title=f'{name}: feature importance')
        fig.tight_layout();fig.savefig(FIGURES/f'08_{name.lower()}_importance.png',dpi=200);plt.show()

    import joblib
    if BEST_MODEL != 'Persistence':
        joblib.dump(fitted[BEST_MODEL],MODELS/'cv_selected_model.joblib')
    print('Finished evaluation. Tables:',TABLES)
    print('Figures:',FIGURES)
else:
    print('No fitted models. Change RUN_MODE to full and use the complete household dataset.')

## 8. Run record and reporting notes

The run summary saves the model setup, available files, selected features and split counts. Report **the count from this run**, rather than assuming all 70 houses have a usable target.

**Interpretation.** A low ten-minute error is *not* a one-hour forecast result. It may reflect strong short-term autocorrelation. State-monitoring temperatures are only proxies for cooking; their feature importance is not a causal conclusion. The 100 µg/m³ cap, if enabled, changes the target and must be mentioned in Methods. The raw (uncapped) run is a worthwhile sensitivity analysis.

**Original code reviewed:** `config.py`, `data_loader.py`, `feature_engineering.py`, `pipeline.py`, `run_linear.py`, `run_lightgbm.py`, `run_xgboost.py`, `tune_lightgbm.py`, `tune_xgboost.py`. This notebook intentionally does not reproduce the original use of test labels for tuning.

In [ ]:
run_summary = {
    'task':'next 10-minute interval indoor PM2.5 prediction',
    'forecast_horizon_minutes':10,
    'target':'PM_MET_IN1 averaged over target bin',
    'predictors_only_from_previous_completed_bin':True,
    'interval':FREQ,
    'minimum_pm_points_per_bin':MIN_PM_POINTS,
    'pm_cap_ug_m3':PM_CAP,
    'feature_level':FEATURE_LEVEL,
    'data_root':str(DATA_ROOT),
    'results_dir':str(RESULTS),
    'file_counts':{k:len(v) for k,v in files.items()},
    'eligible_homes':int(eligible.household_id.nunique()),
    'eligible_rows':int(len(eligible)),
    'split':split_summary,
    'feature_names':features,
    'model_training_completed':bool(RUN_TRAINING),
    'available_model_names': ['Persistence','Linear'] + list(spaces),
    'selected_by_development_cv':BEST_MODEL,
    'test_households_not_used_for_model_selection':True,
    'prior_models_exposed_test_households_in_earlier_experiments':True,
}
(TABLES/'00_run_summary.json').write_text(json.dumps(run_summary,indent=2),encoding='utf-8')
print('Saved summary:',TABLES/'00_run_summary.json')